# QuickCart — Case 1: One Population Mean

**Role:** Junior Data Scientist, Operations Analytics  
**Task:** Plan the number of post-upgrade deliveries needed to determine whether average delivery time differs from 30 minutes.

This notebook corrects and extends the original five-cell practice notebook. The original Z-based calculation is retained and clearly labeled as an approximation.

## 1. Business problem and assumptions

QuickCart promises an average delivery time of **30 minutes**. After a routing-system upgrade, the Operations Manager expects the mean may be **27 minutes** and wants to test whether it **has changed in either direction**. Historical records suggest a delivery-time standard deviation of approximately **8 minutes**.

| Planning quantity | Value | Meaning |
|---|---:|---|
| Reference mean, $\mu_0$ | 30 minutes | Fixed historical benchmark |
| Anticipated mean, $\mu_1$ | 27 minutes | Alternative value used to design the study |
| Absolute MDE, $\delta$ | $|27-30|=3$ minutes | Difference the study aims to detect with target power |
| Planning SD | 8 minutes | Historical **estimate**, not known population SD |
| Significance level, $\alpha$ | 0.05 | Two-sided Type I error rate |
| Desired power, $1-\beta$ | 0.90 | Power at the specified alternative |

**Design assumptions:** Delivery-time observations are independent; the post-upgrade distribution is approximately normal (as specified in the exercise); and the 30-minute benchmark is treated as fixed rather than estimated from a separate historical sample. In actual logistics data, deliveries can be correlated within drivers, shifts, or regions; if so, this simple model may underestimate the required sample size.

**Important:** This is a *one-sample* study: we will sample deliveries **after the upgrade** and compare their mean with 30 minutes. No separate control group is part of the stated design.

## 2. Statistical test and hypotheses

The outcome (delivery time) is continuous. We compare **one new sample mean** to a fixed reference value. Since the population SD is not known and will be estimated from the new sample, a **two-sided one-sample t-test** is appropriate under the stated assumptions.

$$H_0: \mu = 30 \text{ minutes}$$
$$H_a: \mu \ne 30 \text{ minutes}$$

A two-sided alternative is appropriate because the manager asked whether the mean *changed*, not only whether it fell. The anticipated 27-minute mean determines the **planning effect size**, but does **not** change the two-sided hypotheses.

A historical estimate of SD is sufficient for power *planning*. At analysis time, the t-test uses the standard deviation computed from the newly collected observations.

The new routing system cannot be established as the *cause* of any change from this one-sample before/after-benchmark comparison alone.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm
from statsmodels.stats.power import TTestPower

# QuickCart planning inputs
reference_mean = 30.0       # minutes
anticipated_mean = 27.0     # minutes
planning_sd = 8.0          # historical estimated SD (minutes)
alpha = 0.05
target_power = 0.90
alternative = "two-sided"

mde = abs(anticipated_mean - reference_mean)
standardized_effect = mde / planning_sd

print(f"Absolute MDE: {mde:.1f} minutes")
print(f"Standardized effect size |d|: {standardized_effect:.3f}")

Absolute MDE: 3.0 minutes
Standardized effect size |d|: 0.375


## 3. Manual Z-based sample-size approximation

For a two-sided **one-sample mean** normal approximation:

$$n \approx \frac{\left(z_{1-\alpha/2}+z_{1-\beta}\right)^2 \sigma_{\mathrm{planning}}^2}{\delta^2}$$

This is **not** the exact one-sample t-test sample-size formula; it uses normal quantiles. We include it to check the approximate size of the study.

Here, $n$ is the **total number of independent post-upgrade deliveries**, not observations per group. Round the result upward.

In [2]:
z_alpha = norm.ppf(1 - alpha / 2)
z_power = norm.ppf(target_power)  # 1 - beta = desired power

n_normal_approx = ((z_alpha + z_power)**2 * planning_sd**2) / mde**2
n_normal_rounded = int(np.ceil(n_normal_approx))

print(f"z_(1-alpha/2): {z_alpha:.6f}")
print(f"z_(1-beta): {z_power:.6f}")
print(f"Unrounded normal-approximation n: {n_normal_approx:.3f}")
print(f"Rounded normal-approximation n: {n_normal_rounded} deliveries")

z_(1-alpha/2): 1.959964
z_(1-beta): 1.281552
Unrounded normal-approximation n: 74.719
Rounded normal-approximation n: 75 deliveries


## 4. More accurate one-sample t-test planning (`TTestPower`)

Because the analysis will use a one-sample t-test, use **`TTestPower`**, not `TTestIndPower` (which applies to two independent groups). `TTestPower` numerically solves for sample size using the **noncentral t-distribution**.

The `effect_size` input is the standardized mean departure from the null: $|d|=|\mu_1-\mu_0|/\sigma_{\mathrm{planning}}$. For a two-sided test, its magnitude suffices for sample-size planning.

In [3]:
analysis = TTestPower()

n_t_continuous = analysis.solve_power(
    effect_size=standardized_effect,
    nobs=None,
    alpha=alpha,
    power=target_power,
    alternative=alternative,
)
n_t_required = int(np.ceil(n_t_continuous))

achieved_power = analysis.power(
    effect_size=standardized_effect,
    nobs=n_t_required,
    alpha=alpha,
    alternative=alternative,
)

print(f"Unrounded t-test sample size: {n_t_continuous:.3f}")
print(f"Required t-test sample size: {n_t_required} deliveries")
print(f"Calculated achieved power: {achieved_power:.2%}")

Unrounded t-test sample size: 76.666
Required t-test sample size: 77 deliveries
Calculated achieved power: 90.13%


## 5. Compare the two methods

The normal formula is a useful check, but the noncentral-t result is the one to use for this planned t-test. Check the actual t-test power for **both** rounded sample sizes.

In [4]:
comparison = pd.DataFrame({
    "Method": ["Normal approximation", "Noncentral-t planning"],
    "Number of deliveries": [n_normal_rounded, n_t_required],
})
comparison["Power under the t-test model"] = [
    analysis.power(effect_size=standardized_effect, nobs=int(n), alpha=alpha, alternative=alternative)
    for n in comparison["Number of deliveries"]
]
print(comparison.to_string(index=False, formatters={
    "Power under the t-test model": lambda x: f"{x:.2%}"
}))

               Method  Number of deliveries Power under the t-test model
 Normal approximation                    75                       89.35%
Noncentral-t planning                    77                       90.13%


## 6. Sensitivity to the historical SD estimate

The SD of 8 minutes is uncertain. To evaluate whether the plan is robust, repeat the **t-test** calculation with other plausible planning SDs. These alternatives are illustrative assumptions, not observed data.

In [5]:
sensitivity_rows = []
for assumed_sd in [6.0, 8.0, 10.0]:
    assumed_d = mde / assumed_sd
    n_unrounded = analysis.solve_power(
        effect_size=assumed_d, alpha=alpha, power=target_power,
        alternative=alternative
    )
    n_rounded = int(np.ceil(n_unrounded))
    sensitivity_rows.append({
        "Assumed SD (minutes)": assumed_sd,
        "Required deliveries": n_rounded,
        "Power at rounded n": analysis.power(
            effect_size=assumed_d, nobs=n_rounded, alpha=alpha, alternative=alternative
        )
    })

sensitivity = pd.DataFrame(sensitivity_rows)
print(sensitivity.to_string(index=False, formatters={
    "Power at rounded n": lambda x: f"{x:.2%}"
}))

 Assumed SD (minutes)  Required deliveries Power at rounded n
                  6.0                   44             90.00%
                  8.0                   77             90.13%
                 10.0                  119             90.08%


## 7. Recommendation to the Operations Manager

**Recommendation:** Under a two-sided one-sample t-test, a 3-minute detectable change, estimated SD of 8 minutes, $\alpha=0.05$, and 90% desired power, plan for **77 independent, completed post-upgrade deliveries**. The t-test power at this sample size is approximately **90.1%**, provided the planning assumptions hold.

**What this means:** If the true mean delivery time differs from 30 minutes by 3 minutes, this design has about a 90% probability of yielding a statistically significant two-sided test under the assumed model. It does **not** guarantee significance or show that the routing upgrade caused a change.

**Operational checks before collecting data:**

- Ensure the sampled deliveries represent relevant drivers, regions, days, and times; check for clustering or dependence.
- Validate whether the historical 8-minute SD is representative of the new delivery process. If SD may be 10 minutes, the sensitivity analysis shows that more deliveries are needed to preserve 90% power.
- Define the observation window, exclusions, and analysis plan *before* collecting the sample.

**After data collection:** Estimate the actual sample SD, run the prespecified one-sample t-test against 30 minutes, and report the sample mean, 95% confidence interval, effect magnitude, and p-value. No post-upgrade data have been supplied yet, so this notebook performs **planning**, not the final hypothesis test.